<a href="https://colab.research.google.com/github/NaghamZidiah/FlyRank-ML-Internship/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
print("Token loaded successfully!")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

print("Connected successfully!")

rel = "hf://datasets/FlyRank/internship-warehouse"


Token loaded successfully!
Connected successfully!


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [ ]:

distribution_query = f"""
SELECT
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
  AND gsc_impressions > 0
  AND gsc_avg_position IS NOT NULL
LIMIT 100000
"""

distribution_df = con.sql(distribution_query).df()

print("Rows in development sample:", len(distribution_df))
display(distribution_df.describe())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in development sample: 100000


,gsc_impressions,gsc_clicks,gsc_avg_position
count,100000.000000,100000.000000,100000.000000
mean,78.492700,0.225640,13.406244
std,293.657051,0.991199,16.554331
min,1.000000,0.000000,0.000000
25%,5.000000,0.000000,3.555556
50%,19.000000,0.000000,7.000000
75%,68.000000,0.000000,16.760899
max,39003.000000,67.000000,277.000000


### Distribution observations

The development sample shows right-skewed distributions and heavy tails, especially for GSC impressions and clicks. For example, impressions have a median of 19 but a maximum of 39,003, while clicks have a median of 0 but a maximum of 67. Average position also varies widely, with a median of 7.0 and a maximum of 277.

These distributions suggest that a small number of pages have much larger values than the typical page, so averages alone may not represent the data well. Percentiles and bucketed comparisons are useful for the signal tests.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [ ]:
# Signal test #1
# Signal: Higher search volume is associated with higher clicks

signal_1_query = f"""
SELECT
    CASE
        WHEN gsc_impressions BETWEEN 1 AND 99 THEN '1-99'
        WHEN gsc_impressions BETWEEN 100 AND 999 THEN '100-999'
        WHEN gsc_impressions >= 1000 THEN '1000+'
    END AS impression_bucket,
    COUNT(*) AS n,
    AVG(gsc_clicks) AS avg_clicks
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
  AND gsc_impressions > 0
  AND gsc_clicks IS NOT NULL
GROUP BY 1
ORDER BY
    CASE impression_bucket
        WHEN '1-99' THEN 1
        WHEN '100-999' THEN 2
        WHEN '1000+' THEN 3
    END
"""

signal_1_df = con.sql(signal_1_query).df()

display(signal_1_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,impression_bucket,n,avg_clicks
0,1-99,2972453,0.057965
1,100-999,606189,0.800425
2,1000+,32419,5.068787


### Signal test #1: Search volume and clicks

**Signal:** Pages with higher search volume tend to receive more clicks.

**Mini-test:** Pages were grouped into three impression buckets and their average clicks were compared.

**Result:** Average clicks increased from 0.058 for pages with 1–99 impressions, to 0.800 for pages with 100–999 impressions, and 5.069 for pages with 1000+ impressions.

**Verdict: CONFIRMED**

The observed data supports the signal. However, this is an observed association and does not establish that higher impressions cause higher clicks.

In [ ]:
# Signal test #2
# Signal: Better search position is associated with higher CTR

signal_2_query = f"""
SELECT
    CASE
        WHEN gsc_avg_position <= 3 THEN '1-3'
        WHEN gsc_avg_position <= 10 THEN '4-10'
        WHEN gsc_avg_position <= 20 THEN '11-20'
        ELSE '21+'
    END AS position_bucket,
    COUNT(*) AS n,
    AVG(gsc_impressions) AS avg_impressions,
    AVG(gsc_clicks) AS avg_clicks,
    AVG(
        CASE
            WHEN gsc_impressions > 0
            THEN CAST(gsc_clicks AS DOUBLE) / gsc_impressions
        END
    ) AS avg_ctr
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
  AND gsc_impressions > 0
  AND gsc_avg_position IS NOT NULL
  AND gsc_clicks IS NOT NULL
GROUP BY 1
ORDER BY
    CASE position_bucket
        WHEN '1-3' THEN 1
        WHEN '4-10' THEN 2
        WHEN '11-20' THEN 3
        WHEN '21+' THEN 4
    END
"""

signal_2_df = con.sql(signal_2_query).df()

display(signal_2_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n,avg_impressions,avg_clicks,avg_ctr
0,1-3,727362,74.280199,0.282469,0.004756
1,4-10,1456122,94.655608,0.306175,0.003473
2,11-20,519223,56.596118,0.178053,0.002770
3,21+,908354,65.407183,0.085977,0.001289


### Signal test #2: Search position and CTR

**Signal:** Pages with better search positions tend to have higher CTR.

**Mini-test:** Pages were grouped by average search position, and the average CTR was compared across the position buckets.

**Result:** Average CTR was highest for positions 1–3 (0.004756) and decreased across the buckets to 0.003473 for positions 4–10, 0.002770 for positions 11–20, and 0.001289 for positions 21+.

**Verdict: CONFIRMED**

The observed data supports the signal: pages with better average search positions had higher CTR. However, this is an observed association and does not establish that search position alone causes the difference in CTR.

In [ ]:
# Signal test #3
# Signal: Higher sessions are associated with stronger engagement rate

signal_3_query = f"""
SELECT
    CASE
        WHEN ga4_sessions BETWEEN 1 AND 9 THEN '1-9'
        WHEN ga4_sessions BETWEEN 10 AND 49 THEN '10-49'
        WHEN ga4_sessions BETWEEN 50 AND 199 THEN '50-199'
        WHEN ga4_sessions >= 200 THEN '200+'
    END AS sessions_bucket,
    COUNT(*) AS n,
    AVG(ga4_sessions) AS avg_sessions,
    AVG(
        CASE
            WHEN ga4_sessions > 0
            THEN CAST(ga4_engaged_sessions AS DOUBLE) / ga4_sessions
        END
    ) AS avg_engagement_rate
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
  AND ga4_sessions > 0
  AND ga4_engaged_sessions IS NOT NULL
GROUP BY 1
ORDER BY
    CASE sessions_bucket
        WHEN '1-9' THEN 1
        WHEN '10-49' THEN 2
        WHEN '50-199' THEN 3
        WHEN '200+' THEN 4
    END
"""

signal_3_df = con.sql(signal_3_query).df()

display(signal_3_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,sessions_bucket,n,avg_sessions,avg_engagement_rate
0,1-9,385827,1.965829,0.036638
1,10-49,23018,18.060518,0.011290
2,50-199,1437,76.712596,0.005109
3,200+,53,290.283019,0.005702


### Signal test #3: Sessions and engagement rate

**Signal:** Pages with more sessions tend to have a stronger engagement rate.

**Mini-test:** Pages were grouped by GA4 sessions, and the average engagement rate was compared across the session buckets.

**Result:** Average engagement rate decreased from 0.036638 for pages with 1–9 sessions to 0.011290 for 10–49 sessions and 0.005109 for 50–199 sessions. The 200+ bucket increased slightly to 0.005702, but it contained only 53 rows.

**Verdict: OPPOSITE**

The observed data goes against the expected positive relationship. Engagement rate generally decreased as sessions increased, so this signal is not supported by the sample. The small increase in the 200+ bucket should be treated cautiously because that group contains very few rows.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [ ]:
# Flag-linked signal: High-volume pages with weaker positions tend to have lower CTR

flag_test_query = f"""
SELECT
    CASE
        WHEN gsc_avg_position <= 7.5 THEN 'Better than median'
        ELSE 'Worse than median'
    END AS position_group,
    COUNT(*) AS n,
    AVG(gsc_impressions) AS avg_impressions,
    AVG(gsc_clicks) AS avg_clicks,
    AVG(
        CASE
            WHEN gsc_impressions > 0
            THEN CAST(gsc_clicks AS DOUBLE) / gsc_impressions
        END
    ) AS avg_ctr
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
  AND gsc_impressions >= 62
  AND gsc_avg_position IS NOT NULL
  AND gsc_clicks IS NOT NULL
GROUP BY 1
ORDER BY
    CASE position_group
        WHEN 'Better than median' THEN 1
        WHEN 'Worse than median' THEN 2
    END
"""

flag_test_df = con.sql(flag_test_query).df()

display(flag_test_df)

,position_group,n,avg_impressions,avg_clicks,avg_ctr
0,Better than median,545825,281.512441,0.969410,0.003528
1,Worse than median,359647,244.151746,0.514593,0.002352


**Flag-linked signal:** High-volume pages with weaker search positions tend to have lower CTR.

**Mini-test:** Among pages with at least 62 impressions, CTR was compared between pages with a better-than-median position (≤7.5) and pages with a worse-than-median position (>7.5).

**Result:** The better-than-median group had an average CTR of 0.003528, compared with 0.002352 for the worse-than-median group.

**Verdict: CONFIRMED**

The observed data supports the assumption behind this flag-linked signal. High-volume pages with weaker positions had lower CTR in this sample. This is an observed relationship, not proof that changing a page's position or CTR will necessarily produce a specific outcome.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The audit suggests that search volume and search position are useful signals for prioritizing content review, especially when high-volume pages have weaker positions and lower CTR. However, the mixed engagement-rate result shows that not every intuitive signal is reliable, so content teams should use these signals as review indicators rather than final decisions.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.